# Notebook 05 — Build a Semantic Search Engine

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

This is the payoff. We build a small **semantic search engine**: a pile of documents, and a
`search()` function that finds the ones closest in *meaning* to whatever you ask — even when
your words and the document's words don't overlap at all.

Every piece here you've already met: embeddings (Notebook 03) and a vector store (Notebook
04). We're just assembling them.

The next cell installs the two libraries we need (`sentence-transformers` for embeddings, `chromadb` for the vector store) and prints `Ready.` when it finishes. On Colab it takes a minute the first time. If you already ran an earlier notebook in the same session, it will be quick because the libraries are cached.

In [ ]:
%pip install -q sentence-transformers chromadb
print("Ready.")

## Step 1 — A small library of documents

Twenty short documents on a mix of topics: space, cooking, money, health, and tech.

The comments in the next cell group them by topic so you can read them, but those topic labels are never stored. The engine only ever sees the sentence text. We also build a matching list of ids (`doc_0`, `doc_1`, and so on), one per document.

In [2]:
# Our searchable library: 20 short documents grouped into 5 topics.
# The grouping is only for our eyes; the search engine never sees the topic labels.
documents = [
    # Space
    "Astronauts aboard the space station experience weightlessness for months.",
    "A telescope gathers faint light from distant galaxies.",
    "The rover collected rock samples from the surface of Mars.",
    "Comets grow a glowing tail as they near the sun.",
    # Cooking
    "Let the bread dough rise for an hour before baking.",
    "Simmer the tomatoes slowly to deepen the sauce's flavour.",
    "Whisk the eggs until the mixture turns pale and fluffy.",
    "Season the soup with a pinch of salt and fresh herbs.",
    # Money
    "Paying off high-interest debt first saves you the most money.",
    "A diversified portfolio spreads risk across many assets.",
    "Compound interest grows your savings faster over time.",
    "Set aside an emergency fund before you start investing.",
    # Health
    "Regular walking lowers blood pressure and lifts your mood.",
    "Drinking enough water keeps you alert through the afternoon.",
    "Stretching before exercise reduces the chance of injury.",
    "A good night's sleep helps your body repair itself.",
    # Technology
    "The new laptop boots in seconds thanks to its fast drive.",
    "Encryption keeps your messages private from eavesdroppers.",
    "Cloud storage lets you reach your files from any device.",
    "A strong password is long, unusual, and hard to guess.",
]
# Every document needs a unique id. We auto-generate doc_0, doc_1, ... one per document.
ids = [f"doc_{i}" for i in range(len(documents))]
# Confirm how many documents we have before we index them.
print(f"{len(documents)} documents ready.")

20 documents ready.


## Step 2 — Put them in a vector store

The next cell creates a Chroma collection and adds all 20 documents. As they go in, Chroma turns each one into a vector with the MiniLM model for us. We ask it to compare vectors using cosine distance, a good default for sentence meaning. When it finishes it prints `Indexed 20 documents.`

In [3]:
import chromadb  # the vector store
from chromadb.utils import embedding_functions  # helpers that turn text into vectors

# The embedding function Chroma will use to embed each document and each query for us.
# all-MiniLM-L6-v2 is a small, free model that runs on your machine.
minilm_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
# Open a store that saves to disk in the ./search_store folder, so it survives a restart.
client = chromadb.PersistentClient(path="./search_store")
# Start clean: if a 'library' collection is left over from a previous run, remove it.
# The try/except means we don't crash on the first run when there is nothing to delete.
try:
    client.delete_collection("library")
except Exception:
    pass
# Create the collection that holds our documents and their vectors.
# embedding_function: Chroma embeds text for us using the model above.
# metadata={"hnsw:space": "cosine"}: measure closeness by cosine distance, which fits text well.
library = client.create_collection(
    name="library",
    embedding_function=minilm_ef,
    metadata={"hnsw:space": "cosine"},
)
# Add the documents. Chroma embeds each one and stores the text, the id, and the vector.
library.add(ids=ids, documents=documents)
# count() reports how many documents are now stored and searchable.
print(f"Indexed {library.count()} documents.")

/Users/riteshmodi/gits/leanpub_courses/courses/vectors-and-embeddings/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14531.22it/s]

Indexed 20 documents.


## Step 3 — The search function

Three lines of real work: hand the query to the store, get back the closest documents, print
them with a similarity score.

When you run it on the question below (which never mentions encryption or cloud storage), it still returns the tech documents, with the top match near similarity 0.447. Higher similarity means closer in meaning. Keep in mind the engine always hands back its nearest `k` documents, so a low top score means nothing in the library matched well.

In [4]:
# search() takes a question and returns the k closest documents. k defaults to 3.
def search(query, k=3):
    # Chroma embeds the query the same way it embedded the documents, then finds the
    # n_results closest ones. query_texts is a list, so we wrap our single query in [ ].
    result = library.query(query_texts=[query], n_results=k)
    # Echo the question we asked, in quotes, so the output is easy to read.
    print(f"{query!r}\n")
    # Each result has a document and a distance. zip pairs them up so we can loop together.
    # The [0] picks the results for our one query (Chroma can take several queries at once).
    for doc, dist in zip(result["documents"][0], result["distances"][0]):
        # Smaller distance means closer in meaning. similarity = 1 - distance reads more
        # naturally (higher is better). :.3f rounds it to three decimal places.
        print(f"  similarity {1 - dist:.3f}   {doc}")
    print()  # blank line to separate one search's output from the next

# A first demo. The question shares no words with the documents, yet the engine still
# returns the encryption and cloud-storage lines, with a top similarity near 0.447.
# Note the engine always returns its nearest k. A low top score is a sign that nothing
# in the library really matched, not that the search failed.
search("how do I keep my data safe online?")

'how do I keep my data safe online?'

  similarity 0.447   Encryption keeps your messages private from eavesdroppers.
  similarity 0.374   Cloud storage lets you reach your files from any device.
  similarity 0.174   A strong password is long, unusual, and hard to guess.



## Step 4 — The magic: no shared words

Watch each query find the right documents even though it shares almost no words with them.

Each query below lands on documents from the right topic even though it shares almost no words with them. The top score for a good match tends to sit around 0.4 to 0.5 here ("growing my money for the future" tops out near 0.495). The trailing comments note which topic each query should find.

In [5]:
search("what helps me relax and feel rested?")   # -> sleep / walking / water
search("growing my money for the future")         # -> investing / compound interest
search("exploring other planets")                 # -> Mars rover / telescope / comets

'what helps me relax and feel rested?'

  similarity 0.413   A good night's sleep helps your body repair itself.
  similarity 0.333   Drinking enough water keeps you alert through the afternoon.
  similarity 0.247   Regular walking lowers blood pressure and lifts your mood.

'growing my money for the future'

  similarity 0.495   Compound interest grows your savings faster over time.
  similarity 0.439   Set aside an emergency fund before you start investing.
  similarity 0.383   Paying off high-interest debt first saves you the most money.

'exploring other planets'

  similarity 0.405   The rover collected rock samples from the surface of Mars.
  similarity 0.357   A telescope gathers faint light from distant galaxies.
  similarity 0.242   Comets grow a glowing tail as they near the sun.



## Step 5 — Your turn, and recap

**Try it:** call `search("...")` with your own questions. Try one that shares zero words with
any document and watch it still find the right topic.

**Recap**
- A semantic search engine = embeddings + a vector store + a tiny `search()` wrapper.
- It matches meaning, so it finds documents that share *ideas*, not just words.
- You built the whole thing from parts you already understood.

**Next (Notebook 06):** a mini challenge — you build a Smart FAQ Finder mostly on your own,
with hints if you need them.

## Practice — Your Turn

Three short exercises using the `search()` function and the `library` you just built. Read each task, make your prediction, then run the answer cell underneath.

### Exercise 1 — A health question with no shared words

Ask the engine a plain health question, "how do I stay healthy?". None of the health documents contain the words "stay" or "healthy", so the engine has to match on meaning alone.

Before you run the answer cell, predict: which topic group do you expect to win, and roughly how high will the top similarity be?

Try it yourself, then run the answer cell below.

In [6]:
# Answer
# Ask a health question that shares no words with the health documents.
search("how do I stay healthy?")
# Read the three results above. The walking, sleep, and water lines should rise
# to the top, even though none of them contain the words "stay" or "healthy".
# That is meaning-based matching at work: close ideas, different words.

'how do I stay healthy?'

  similarity 0.260   Season the soup with a pinch of salt and fresh herbs.
  similarity 0.167   Stretching before exercise reduces the chance of injury.
  similarity 0.161   Regular walking lowers blood pressure and lifts your mood.



### Exercise 2 — Add your own documents

So far the library holds 20 documents. Here you add a few of your own about gardening, a topic the library does not yet cover. Then you search for it and watch one of your new documents take the top spot.

We give the new documents brand-new ids (`mine_0`, `mine_1`, ...) so they never clash with the existing `doc_0`, `doc_1`, ... ids. We also check first whether they are already in the library, so you can run this cell more than once without an error.

Try it yourself, then run the answer cell below.

In [7]:
# Answer
# Three new documents on gardening, a topic the library did not have before.
my_docs = [
    "Water your tomato plants early in the morning to avoid scorching the leaves.",
    "Pull weeds while they are small so they do not steal water from your crops.",
    "Add compost to the soil each spring to feed your vegetable garden.",
]
# Brand-new ids that cannot collide with the existing doc_0, doc_1, ... ids.
my_ids = ["mine_0", "mine_1", "mine_2"]

# Check which of our ids are already stored, so re-running this cell is safe.
existing = set(library.get(ids=my_ids)["ids"])  # ids already in the library
# Keep only the documents whose id is not stored yet.
to_add_docs = [d for d, i in zip(my_docs, my_ids) if i not in existing]
to_add_ids = [i for i in my_ids if i not in existing]

# Add the new documents only if there is something new to add.
if to_add_ids:
    library.add(ids=to_add_ids, documents=to_add_docs)  # Chroma embeds and stores them
    print(f"Added {len(to_add_ids)} new documents.")
else:
    print("New documents already present, skipping the add.")

# Confirm the library grew.
print(f"Library now holds {library.count()} documents.\n")

# Search for the new topic. One of your gardening lines should be the top hit.
search("tips for growing vegetables in my backyard")

Added 3 new documents.
Library now holds 23 documents.

'tips for growing vegetables in my backyard'

  similarity 0.556   Add compost to the soil each spring to feed your vegetable garden.
  similarity 0.463   Pull weeds while they are small so they do not steal water from your crops.
  similarity 0.306   Water your tomato plants early in the morning to avoid scorching the leaves.



### Exercise 3 — The nonsense query

The engine always hands back its nearest `k` documents, even when nothing in the library really fits. To see this, search for "purple dinosaur tax law", a phrase that matches none of the topics well.

Predict: how high do you think the top similarity will be? Compare it to the good matches you saw earlier, which sat around 0.4 to 0.5.

Try it yourself, then run the answer cell below.

In [8]:
# Answer
# A query that does not really belong to any topic in the library.
query = "purple dinosaur tax law"
# Run the normal search so we can see the three documents it picks anyway.
search(query)

# Now look at just the top similarity on its own, so we can judge how weak it is.
result = library.query(query_texts=[query], n_results=1)  # ask for the single best match
top_distance = result["distances"][0][0]                   # distance of that best match
top_similarity = 1 - top_distance                          # turn distance into similarity
# Report the number and what it tells us.
print(f"Top similarity for a nonsense query: {top_similarity:.3f}")
# A low score (well under about 0.2) means nothing in the library truly matched.
# The engine did not fail. It simply returned its nearest neighbours, as it always does.

'purple dinosaur tax law'

  similarity 0.192   Compound interest grows your savings faster over time.
  similarity 0.128   Comets grow a glowing tail as they near the sun.
  similarity 0.119   Pull weeds while they are small so they do not steal water from your crops.

Top similarity for a nonsense query: 0.192
